# 7. MODEL DEPLOYMENT PREPARATION
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../models/churn_model_YYYYMMDD.joblib`, `../data/processed/churn_features_YYYYMMDD.parquet`, and `../data/processed/churn_explainability_YYYYMMDD.parquet`

**OUTPUT:** `../models/churn_scoring_package_YYYYMMDD.joblib`, `../src/models/churn_scoring.py`, and `../data/processed/churn_inference_smoke_test_YYYYMMDD.parquet`

*A reusable scoring package with deployment-ready metadata, inference helpers, and a smoke-tested prediction example.*


---
## 7.1. STARTING SITUATION


The model is now trained, diagnosed and explained. Before orchestration can consume it, the project needs a stable inference package that reproduces the same feature alignment, risk-tier logic and retention-rule outputs every time the model runs.


---
## 7.2. NOTEBOOK OBJECTIVE


- **Business objective:** package the churn model so daily scoring can run reliably and produce outputs that downstream systems can consume without manual intervention.
- **Analytical objective:** create a deployment-ready artifact, a plain Python scoring module, and a smoke test proving end-to-end inference works on recent customer snapshots.


In [1]:
import json
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
logger = logging.getLogger('nb07_deployment_prep')
logger.info('NB07 started: model deployment preparation.')


2026-05-02 00:38:34,784 | INFO | NB07 started: model deployment preparation.


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
SRC_MODELS_DIR = PROJECT_ROOT / 'src' / 'models'
SRC_MODELS_DIR.mkdir(parents=True, exist_ok=True)

run_date_tag = datetime.now(ZoneInfo('Europe/Paris')).strftime('%Y%m%d')
model_path = sorted(MODELS_DIR.glob('churn_model_*.joblib'))[-1]
feature_path = sorted(PROCESSED_DIR.glob('churn_features_*.parquet'))[-1]
explainability_path = sorted(PROCESSED_DIR.glob('churn_explainability_*.parquet'))[-1]
scoring_package_path = MODELS_DIR / f'churn_scoring_package_{run_date_tag}.joblib'
smoke_test_path = PROCESSED_DIR / f'churn_inference_smoke_test_{run_date_tag}.parquet'
scoring_module_path = SRC_MODELS_DIR / 'churn_scoring.py'


In [3]:
package = joblib.load(model_path)
feature_df = pd.read_parquet(feature_path)
explainability_df = pd.read_parquet(explainability_path)
model = package['model']
feature_columns = package['feature_columns']

inference_metadata = {
    'model_name': package['model_name'],
    'feature_columns': feature_columns,
    'risk_thresholds': {'low_max': 0.40, 'medium_max': 0.70},
    'retention_rules': {
        'high': {'base_discount_pct': 25, 'vip_discount_pct': 30, 'free_shipping': True},
        'medium': {'base_discount_pct': 12, 'free_shipping': True},
        'low': {'base_discount_pct': 0, 'free_shipping': False},
    },
    'test_snapshot_keys': package['test_snapshot_keys'],
}
deployment_bundle = {'metadata': inference_metadata, 'model_package': package}
joblib.dump(deployment_bundle, scoring_package_path)
logger.info('Scoring package saved to %s', scoring_package_path)


2026-05-02 00:38:35,671 | INFO | Scoring package saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/models/churn_scoring_package_20260502.joblib


In [4]:
scoring_module = (
    "import logging\n\n"
    "import joblib\n"
    "import pandas as pd\n\n"
    "logger = logging.getLogger(__name__)\n\n"
    "def apply_risk_tier(probability: float) -> str:\n"
    "    if probability > 0.70:\n"
    "        return 'HIGH'\n"
    "    if probability >= 0.40:\n"
    "        return 'MEDIUM'\n"
    "    return 'LOW'\n\n"
    "def score_dataframe(df: pd.DataFrame, package_path: str) -> pd.DataFrame:\n"
    "    bundle = joblib.load(package_path)\n"
    "    package = bundle['model_package']\n"
    "    model = package['model']\n"
    "    feature_columns = package['feature_columns']\n\n"
    "    encoded = pd.get_dummies(df.copy(), columns=['customer_state'], dtype=float)\n"
    "    encoded = encoded.reindex(columns=feature_columns, fill_value=0.0)\n"
    "    probabilities = model.predict_proba(encoded)[:, 1]\n\n"
    "    scored = df.copy()\n"
    "    scored['churn_probability'] = probabilities\n"
    "    scored['risk_tier'] = [apply_risk_tier(value) for value in probabilities]\n"
    "    return scored\n"
)
scoring_module_path.write_text(scoring_module, encoding='utf-8')
logger.info('Scoring module written to %s', scoring_module_path)


2026-05-02 00:38:35,678 | INFO | Scoring module written to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/src/models/churn_scoring.py


In [5]:
leakage_columns = [
    'customer_unique_id', 'snapshot_key', 'snapshot_date', 'first_purchase_timestamp',
    'last_purchase_timestamp', 'future_orders_90d', 'future_revenue_90d', 'churn_90d_label'
]
latest_snapshot_key = sorted(feature_df['snapshot_key'].unique())[-1]
latest_snapshot_df = feature_df[feature_df['snapshot_key'] == latest_snapshot_key].copy()
model_input_df = latest_snapshot_df[[c for c in feature_df.columns if c not in leakage_columns]].copy()

namespace = {}
exec(scoring_module_path.read_text(encoding='utf-8'), namespace)
smoke_test_scored = namespace['score_dataframe'](model_input_df.head(500).copy(), str(scoring_package_path))
smoke_test_scored.insert(0, 'customer_unique_id', latest_snapshot_df.head(500)['customer_unique_id'].to_numpy())
smoke_test_scored.insert(1, 'snapshot_key', latest_snapshot_df.head(500)['snapshot_key'].to_numpy())
smoke_test_scored.to_parquet(smoke_test_path, index=False)
logger.info('Inference smoke test saved to %s', smoke_test_path)
smoke_test_scored.head()


2026-05-02 00:38:35,730 | INFO | Inference smoke test saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_inference_smoke_test_20260502.parquet


,customer_unique_id,snapshot_key,customer_state,total_orders,delivered_orders,total_items,total_payment_value,total_item_price,total_freight_value,avg_order_value,...,is_repeat_customer,active_last_30d,active_last_60d,delivered_order_share,credit_card_share_total,boleto_share_total,voucher_share_total,debit_card_share_total,churn_probability,risk_tier
209817,0000366f3b9a7992bf8c76cfdf3221e2,20180701,SP,1,1,1.0,141.90,129.90,12.00,141.90,...,0,0,1,1.0,1.0,0.0,0.0,0.0,0.546753,MEDIUM
209818,0000b849f77a49e4a4ce2b2a4ca5be3f,20180701,SP,1,1,1.0,27.19,18.90,8.29,27.19,...,0,0,1,1.0,1.0,0.0,0.0,0.0,0.685607,MEDIUM
209819,0004bd2a26a76fe21f786e4fbd80607f,20180701,SP,1,1,1.0,166.98,154.00,12.98,166.98,...,0,0,0,1.0,1.0,0.0,0.0,0.0,0.647283,MEDIUM
209820,00050ab1314c0e55a6ca13cf7181fecf,20180701,SP,1,1,1.0,35.38,27.99,7.39,35.38,...,0,0,0,1.0,0.0,1.0,0.0,0.0,0.782305,HIGH
209821,000949456b182f53c18b68d6babc79c1,20180701,SP,1,1,1.0,82.05,64.89,17.16,82.05,...,0,0,0,1.0,0.0,1.0,0.0,0.0,0.621534,MEDIUM


---
## 7.3. NOTEBOOK CLOSURE


The deployment-preparation stage now has a reusable scoring package, a plain Python module for inference, and a smoke test proving that end-to-end prediction works outside the original training notebook.

The next notebook should translate these artifacts into an orchestration blueprint that n8n can run on a daily schedule with business-rule outputs already attached.
